### Plot Pairwise Correlations for Spheroid Aggregated data

In [ ]:
# --- repo path bootstrap (added by the port) ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, external, require)
from utils.panels import save_panel

import pandas as pd
import numpy as np
import os

from matplotlib.patches import Patch
from sklearn.metrics.pairwise import cosine_similarity
# The fingerprint/ranking cells call `cos_sim`; upstream it came from an earlier
# interactive session and was never imported here. It is plain cosine_similarity.
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

# Plotting
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns; sns.set_style("white")

# --- helpers the source notebook used but never defined -----------------------
# `parse_channel` is recovered verbatim from
# spher_colo52_v1/3_Figure4/PairwiseCorrelations/plot_cluster_signature.py (l.77, l.200),
# which the triage had wrongly excluded as exploratory. See KNOWN_ISSUES.md.
_CHANNELS = {'HOECHST', 'PHAandWGA', 'MITO', 'SYTO', 'CONC'}


def parse_channel(feat: str) -> str:
    parts = feat.split('_')
    for p in parts:
        if p in _CHANNELS:
            return p
    return 'none'


# `normalize_feat` is called by the Fig 5f cells but is defined NOWHERE in
# colopaint3D, colopaint3D_fork or colopaint3D_AZ — it survived only in a live
# kernel. RECONSTRUCTED, not recovered, and enabled on the author's instruction.
#
# Evidence it is the right rule: the 2D and 3D feature tables differ only in an
# "illum" prefix glued to the channel token. 983 of 1093 2D names carry it
# (illumHOECHST x270, illumCONC x228, illumMITO x219, illumSYTO x216,
# illumPHAandWGA x198); none of the 598 3D names do. Exact name overlap is 92
# untouched and 467 once the prefix is stripped. It is also what makes
# parse_channel work on 2D names at all, since _CHANNELS holds the bare forms.
import re

_ILLUM_RE = re.compile(r"illum(?=" + "|".join(sorted(_CHANNELS, key=len, reverse=True)) + ")")


def normalize_feat(feat: str) -> str:
    """Strip the 2D tables' `illum` channel prefix so names match the 3D tables.

    RadialDistribution_ZernikePhase_illumPHAandWGA_9_3_cytoplasm
        -> RadialDistribution_ZernikePhase_PHAandWGA_9_3_cytoplasm
    """
    return _ILLUM_RE.sub("", feat)


# Fig 5f compares 2D against 3D single-cell aggregates. It is not defined for the
# MIP runs — 5-FU does not clear grit there, so the fingerprint cells have no
# 5-FU row to index. Gate on the data type rather than letting them fail.
import os as _os
FIG5F_AVAILABLE = _os.environ.get('COLOPAINT3D_DATA_TYPE', 'aggregates') in ('2D', 'aggregates')

# Set current working directory


In [ ]:
# Save the data
ImagesOut = str(figdir('Fig4')) + '/'

if not os.path.exists(ImagesOut): 
        os.makedirs(ImagesOut)

# Remove all previous PNGs in the output directory
import glob
for f in glob.glob(os.path.join(ImagesOut, '*.png')):
    os.remove(f)
print(f'Cleaned up PNGs in {ImagesOut}')

In [ ]:
# Set up the plotting parameters
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
dpi = 300
figformat = 'pdf'

In [ ]:
# Parameters. run_all.py sweeps these via the environment so one notebook can
# emit every combination; the defaults keep interactive use unchanged.
import os
cell_line = os.environ.get('COLOPAINT3D_CELL_LINE', 'HCT116')
data_type = os.environ.get('COLOPAINT3D_DATA_TYPE', 'aggregates')

# Paper panel for this (cell_line, data_type). One run per combination; save_panel
# routes each to its figure. The clustermap appears in Fig4, Fig5 and both supplements.
CLUSTERMAP_PANEL = {
    ("HCT116", "MIP"):        "Fig4e",
    ("HCT116", "aggregates"): "Fig4f",
    ("HCT116", "2D"):         "Fig5c",
    ("HT29",   "MIP"):        "SupplFig4e",
    ("HT29",   "aggregates"): "SupplFig4f",
    ("HT29",   "2D"):         "SupplFig5b",
}
# The 2D-minus-3D difference map depends only on the cell line.
DIFFERENCE_PANEL = {"HCT116": "Fig5d", "HT29": "SupplFig5c"}


In [ ]:
# Load the data
dir = str(profiles("exp1_main", "")) + "/"
data =  pd.read_parquet(('{}grit_data_{}_{}.parquet').format(dir, data_type, cell_line))
data.dropna(axis='columns', how='all', inplace=True)

In [ ]:
# Some function definitions

def list_features(df):
    # List features
    list_of_selected_features = list(df.columns.values)
    list_of_metadata = list(df.columns[df.columns.str.contains("Metadata_")])
    list_of_selected_features = list(set(list_of_selected_features) - set(list_of_metadata))
    
    return list_of_selected_features, list_of_metadata

def analyze_pathways(dataset, filter_criteria="Metadata_grit > 1.96", 
    correlation_method='cosine'):
    
    # Prepare and filter the dataset
    t = dataset.copy()
    meta = t[list_features(t)[1]]
    t = t.query('Metadata_pert_type == "trt"')
    t = t.groupby('Metadata_pert_name').median(numeric_only=True)
    t.drop(['Metadata_cmpd_conc'], axis=1, inplace=True)
    t = t.query(filter_criteria)
    data = t[list_features(t)[0]]

    # Calculate similarity matrix
    if correlation_method == 'pearson':
        # Use correlation
        sim_matrix = data.T.corr()
        sim_name = 'cor'
    elif correlation_method == 'cosine':
        # Use cosine similarity
        sim_matrix = pd.DataFrame(
            cosine_similarity(data),
            index=data.T.columns, columns=data.T.columns
        )
        sim_name = 'cosine_sim'
    else:
        raise ValueError("Invalid correlation_method. Use 'pearson' or 'cosine'.")

    # Reshape and map compound names
    m = sim_matrix.stack().to_frame(name=sim_name).reset_index(names=['pert_x', 'pert_y'])
    pert_to_name = dict(zip(meta.Metadata_pert_name, meta.Metadata_name))
    m['name_x'] = m['pert_x'].map(pert_to_name)
    m['name_y'] = m['pert_y'].map(pert_to_name)
    similarity_pivot_table = pd.pivot_table(m, values=sim_name, index='name_y', columns='name_x', aggfunc='max')

    return similarity_pivot_table

def calculate_similar_pairs(similarity_pivot_table):
    # Exclude diagonal elements
    _arr = similarity_pivot_table.to_numpy(dtype=float, copy=True)
    np.fill_diagonal(_arr, np.nan)
    p_no_diag = pd.DataFrame(_arr, index=similarity_pivot_table.index,
                             columns=similarity_pivot_table.columns)

    # Convert to long format
    similarities = p_no_diag.unstack().reset_index()
    similarities.columns = ['Drug1', 'Drug2', 'Similarity']

    # Clean up the dataframe
    similarities['Pair'] = similarities.apply(
        lambda row: tuple(sorted([row['Drug1'], row['Drug2']])), axis=1)
    similarities = similarities.drop_duplicates(subset='Pair')
    similarities = similarities.dropna(subset=['Similarity'])
    similarities = similarities.sort_values(by='Similarity', ascending=False)

    return similarities


### Pairwise Correlations

In [ ]:
## Prepare the metadata for the Pairwise calculation
dataset_pw  = data.copy()

# Create a dictionary of pathways and their associated compounds.
compounds = dataset_pw['Metadata_name'].unique()
pathways_dict = {cmpd: dataset_pw[dataset_pw['Metadata_name'] == cmpd]['Metadata_pathway'].unique()[0] for cmpd in compounds}
pathways = pd.Series(['MAPK','Cell Cycle', 'DNA Damage', 
                      'PI3K/Akt/mTOR', 'Epigenetics', 'Stem Cells & Wnt', 
                      'Angiogenesis', 'Protein Tyrosine Kinase', 
                      'Apoptosis', 'JAK/STAT', 'Cytoskeletal Signaling', 
                      'TGF-beta/Smad', 'Others', 'Proteases'])

# Create a color map for the pathways
colors = sns.color_palette('tab20', len(pathways))
lut = dict(zip(pathways, colors))
col_colors = pd.Series(pathways_dict).map(lut)
col_colors = col_colors.dropna()


In [ ]:
# Calculate the similarity matrix
similarity_pivot_table = analyze_pathways(dataset_pw, filter_criteria="Metadata_grit > 1.96", correlation_method='cosine')

# Plot the clustermap
ax = sns.clustermap(
    similarity_pivot_table,
    cmap='RdBu_r',
    xticklabels=1,
    yticklabels=1,
    cbar_pos=(0.01, 0.9, 0.01, 0.18),
    tree_kws=dict(colors='#ddd'),
    cbar_kws=dict(shrink=0.2),
    method='ward',
    dendrogram_ratio=0.1,
    col_colors=col_colors,
    vmin = -0.25,
    vmax = 1
)

# Add legend
handles = [Patch(facecolor=lut[name]) for name in lut]
plt.legend(handles, lut, title='Pathway',
            bbox_to_anchor=(1.2, 1), bbox_transform=plt.gcf().transFigure, loc='upper right')

plt.title(f'{cell_line} - {data_type} - Pathway Similarity')

plt.show()

save_panel(ax.figure, CLUSTERMAP_PANEL[(cell_line, data_type)],
           data=similarity_pivot_table,
           caption=f"Pairwise compound similarity clustermap, {cell_line} {data_type}",
           notebook="analysis/3_Figure4/3_PairwiseCorrelations.ipynb")

In [ ]:
# List the most similar pairs
similarities = calculate_similar_pairs(similarity_pivot_table)

# Display the top 10 most similar pairs
top_10_similar_pairs = similarities.head(10)
print(("Top 10 most similar pairs for {}_{}:").format(cell_line, data_type))
print(top_10_similar_pairs[['Drug1', 'Drug2', 'Similarity']])

### Now compare 2D & 3D

In [ ]:
dir = str(profiles("exp1_main", "")) + "/"

cell_line = os.environ.get('COLOPAINT3D_CELL_LINE', 'HCT116')  # 'HCT116' or 'HT29'
data_types = ['2D', 'aggregates']

In [ ]:
# Set up a dictionary
sim = {}

for data_type in data_types:
    data =  pd.read_parquet(('{}grit_data_{}_{}.parquet').format(dir, data_type, cell_line))
    data.dropna(axis='columns', how='all', inplace=True)
    
    sim[data_type] = calculate_similar_pairs(analyze_pathways(data))
    sim[data_type].sort_values(by=['Drug1', 'Drug2'], ascending=True, inplace=True)
    sim[data_type]['Pair'] = sim[data_type].apply(lambda row: tuple(sorted([row['Drug1'], row['Drug2']])), axis=1)

# Match the pairs in 2D with those in 3D
sim2D = sim['2D'].copy()
sim3D = sim['aggregates'].copy()

# Merge the two dataframes on the Pair column
sim2D['Match'] = sim2D['Pair'].isin(sim3D['Pair'])

# restored by the port — see KNOWN_ISSUES.md
data_2D = pd.read_parquet(profiles('exp1_main', f'grit_data_2D_{cell_line}.parquet')).dropna(axis='columns', how='all')
similarities = sim2D.merge(sim3D, on=['Pair', 'Drug1', 'Drug2'], how='inner', suffixes=('_2D', '_3D'))

similarities['difference'] = similarities['Similarity_3D'] - similarities['Similarity_2D']
similarities['max'] = similarities[['Similarity_2D', 'Similarity_3D']].max(axis=1)

# Similarity_3D - Similarity_2D
differences = similarities['difference']

# Identify the 2.5% (lower) and 97.5% (upper) quantiles
lower_thresh = differences.quantile(0.025)
upper_thresh = differences.quantile(0.975)

# Mark whether each difference is in the extreme 5% region
similarities['tail'] = (
    (differences < lower_thresh) |
    (differences > upper_thresh)
)


In [ ]:
## Plot the differences in a scatterplot

subset = similarities.query('tail == True & max > 0.5')

plt.figure(figsize=(12, 12))
ax = sns.scatterplot(data=similarities, x='difference', y='max', c='gray', alpha=0.5)

sns.scatterplot(data=subset.query('difference < 0 '), x='difference', y='max', c='blue', alpha=0.5)
sns.scatterplot(data=subset.query('difference > 0 '), x='difference', y='max', c='red', alpha=0.5)

# The annotation are overlapping, so we can try to spread them out
for i, row in subset.iterrows():
    text = f"{row['Pair'][0]} vs {row['Pair'][1]}"
    plt.text(row['difference'], row['max'],text, fontsize=8, rotation=45)


plt.axhline(0.5, color='gray', linestyle='--')
plt.axvline(0, color='gray', linestyle='--')
plt.xlabel('Difference in Similarity (2D versus 3D)')
plt.ylabel('Highest Similarity for a given drug pair')
plt.title(f'{cell_line} - Comparison of Cosine Similarity Scores between 2D and 3D \n upper_thresh = {upper_thresh:.2f}; lower_thresh = {lower_thresh:.2f}')

# Add text to the plot
plt.text(-0.68, 1, '2D > 3D', color='darkblue', fontsize=12) # Left: 2D > 3D
plt.text(0.56, 1, '3D > 2D', color='darkred', fontsize=12) # Right: 3D > 2D

# Grab the figure BEFORE plt.show(): under the inline backend show() closes it, so a
# later plt.gcf() would hand back a fresh empty figure and save a blank page.
fig = ax.get_figure()

plt.show()

# Save the plot
save_panel(fig, DIFFERENCE_PANEL[cell_line],
           data=similarities,
           caption=f"2D minus 3D pairwise similarity, {cell_line}",
           notebook="analysis/3_Figure4/3_PairwiseCorrelations.ipynb")


### Fluorouracil Similarities

In [ ]:

# Find Fluorouracil's full name in the dataset (case-insensitive)
fluor_matches = dataset_pw[dataset_pw['Metadata_name'].str.contains('fluor', case=False, na=False)]['Metadata_name'].unique()
print("Fluorouracil matches in dataset:", fluor_matches)

# Check if it passed the grit filter and made it into the similarity matrix
fluor_in_matrix = [m for m in fluor_matches if m in similarity_pivot_table.index]
print("Found in similarity matrix:", fluor_in_matrix)


In [ ]:

# If Fluorouracil didn't pass grit > 1.96, recompute with no filter so it's included
if fluor_in_matrix:
    fluor_name = fluor_in_matrix[0]
    cosine_pt = similarity_pivot_table
else:
    print("Fluorouracil not in grit-filtered matrix — recomputing without grit filter")
    cosine_pt = analyze_pathways(dataset_pw, filter_criteria="Metadata_grit == Metadata_grit", correlation_method='cosine')
    fluor_name = fluor_matches[0]

# Pearson pivot table (same filter as cosine)
filter_used = "Metadata_grit > 1.96" if fluor_in_matrix else "Metadata_grit == Metadata_grit"
pearson_pt = analyze_pathways(dataset_pw, filter_criteria=filter_used, correlation_method='pearson')

# Extract Fluorouracil row and drop self-similarity
cosine_fluor = cosine_pt.loc[fluor_name].drop(labels=fluor_name, errors='ignore').sort_values(ascending=False)
pearson_fluor = pearson_pt.loc[fluor_name].drop(labels=fluor_name, errors='ignore').sort_values(ascending=False)

# Combine into one table
fluor_sim = pd.DataFrame({
    'Cosine Similarity': cosine_fluor,
    'Pearson Correlation': pearson_fluor
}).sort_values('Cosine Similarity', ascending=False)

print(f"\nSimilarities to {fluor_name} (sorted by cosine):")
fluor_sim


### Morphological Fingerprints: Example Compounds

In [ ]:
# Channel color palette (shared across 2D/3D plots)
CHANNEL_PAL = {
    'HOECHST':           '#9467bd',
    'PHAandWGA':         '#bcbd22',
    'MITO':              '#8c564b',
    'SYTO':              '#1f77b4',
    'CONC':              '#ff7f0e',
    'AreaShape':         '#2ca02c',
    'Granularity':       '#d62728',
    'Neighbors':         '#e377c2',
    'Correlation':       '#17becf',
    'RadialDistribution':'#aec7e8',
    'Other':             '#7f7f7f',
    'none':              '#7f7f7f',
}

# Target drugs to display (Metadata_name → display label)
TARGET = {
    'Fluor': '5-FU',
    'Olapa': 'Olaparib',
    'etop':  'Etoposide',
    'Nutli': 'Nutlin',
    'AMG23': 'AMG232',
}
POSCONS = ['etop']


def get_lowest_passing_profiles(dataset, target_keys, extra_names=None, grit_threshold=1.96):
    """
    Returns (fingerprints_df, z_all) where:
    - fingerprints_df: z-scored profile for each drug at its lowest passing concentration
    - z_all:          z-scored profiles for ALL passing concentrations (for ranking)
    Z-scores are computed against all passing-grit profiles in the dataset.
    """
    # --- filter to trt and optionally specified pos_cons ---
    mask = dataset['Metadata_pert_type'] == 'trt'
    if extra_names:
        mask |= (
            (dataset['Metadata_pert_type'] == 'pos_con') &
            dataset['Metadata_name'].isin(extra_names)
        )
    t = dataset[mask].copy()

    feats, meta_cols = list_features(t)

    # groupby pert_name → median (mirrors analyze_pathways)
    t_grp = t.groupby('Metadata_pert_name').median(numeric_only=True)
    # restore name and conc from per-row metadata (they're constant per pert_name)
    meta_lookup = (
        t[['Metadata_pert_name', 'Metadata_name', 'Metadata_cmpd_conc']]
        .drop_duplicates('Metadata_pert_name')
        .set_index('Metadata_pert_name')
    )
    name_map = meta_lookup['Metadata_name']
    conc_map = meta_lookup['Metadata_cmpd_conc']

    # filter to grit-passing concentrations
    passing = t_grp[t_grp['Metadata_grit'] > grit_threshold]
    feat_data = passing[feats]

    # z-score against all passing profiles
    z = (feat_data - feat_data.mean()) / feat_data.std()
    z['_name'] = passing.index.map(name_map)
    z['_conc'] = passing.index.map(conc_map)

    # pick lowest passing concentration per target drug
    rows = {}
    for key, label in target_keys.items():
        subset = z[z['_name'] == key].sort_values('_conc')
        if len(subset) == 0:
            print(f"WARNING: {label} ({key}) has no concentration passing grit > {grit_threshold}")
            continue
        chosen = subset.iloc[0]
        conc   = chosen['_conc']
        rows[f"{label}\n({conc} µM)"] = subset[feats].iloc[0]

    fp_df = pd.DataFrame(rows).T
    return fp_df, z


In [ ]:
# Guarded: this cell is part of the Fig 5f fingerprint comparison, which needs
# `normalize_feat` to match 2D feature names to 3D. That helper is missing from
# the source (see KNOWN_ISSUES.md), so the code is kept verbatim but not run.
if FIG5F_AVAILABLE:
    # ── 2D Morphological Fingerprints ─────────────────────────────────────────────
    fp_2D, z_2D = get_lowest_passing_profiles(data_2D, TARGET, POSCONS)

    # normalize illum prefix so feature names match 3D
    fp_2D.columns = [normalize_feat(c) for c in fp_2D.columns]

    # top 40 features by std across the 5 compounds
    top_feats_2D = fp_2D.std(axis=0).nlargest(40).index
    plot_2D = fp_2D[top_feats_2D]

    # channel row colors
    channels_2D     = [parse_channel(f) for f in top_feats_2D]
    row_colors_2D   = pd.Series(channels_2D, index=top_feats_2D).map(CHANNEL_PAL)

    g = sns.clustermap(
        plot_2D.T,
        cmap='RdBu_r', center=0, vmin=-3, vmax=3,
        row_colors=row_colors_2D,
        figsize=(8, 14),
        dendrogram_ratio=(0.1, 0.2),
        cbar_pos=(1.02, 0.3, 0.03, 0.3),
        xticklabels=True, yticklabels=True,
    )
    g.ax_heatmap.set_title('2D: Morphological Fingerprints (top 40 features)', pad=12)
    g.ax_heatmap.set_xlabel('Compound')
    save_panel(g.figure, "Fig5f_2D", data=plot_2D,
               caption="2D morphological fingerprints, 5-FU and olaparib",
               notebook="analysis/3_Figure4/3_PairwiseCorrelations.ipynb")
    plt.show()

    # Legend for channel colors
    from matplotlib.patches import Patch
    handles = [Patch(color=v, label=k) for k, v in CHANNEL_PAL.items() if k in channels_2D]
    plt.figure(figsize=(3, 3))
    plt.legend(handles=handles, title='Channel', loc='center', frameon=False)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    print("2D chosen concentrations:")
    for row_label in fp_2D.index:
        print(" ", row_label.replace("\n", " "))
else:
    print("SKIP Fig5f section: normalize_feat unavailable — see KNOWN_ISSUES.md")


In [ ]:
# Guarded: this cell is part of the Fig 5f fingerprint comparison, which needs
# `normalize_feat` to match 2D feature names to 3D. That helper is missing from
# the source (see KNOWN_ISSUES.md), so the code is kept verbatim but not run.
if FIG5F_AVAILABLE:
    # ── 3D Morphological Fingerprints ─────────────────────────────────────────────
    fp_3D, z_3D = get_lowest_passing_profiles(dataset_pw, TARGET, POSCONS)

    # top 40 features by std across the 5 compounds
    top_feats_3D = fp_3D.std(axis=0).nlargest(40).index
    plot_3D = fp_3D[top_feats_3D]

    # channel row colors
    channels_3D   = [parse_channel(f) for f in top_feats_3D]
    row_colors_3D = pd.Series(channels_3D, index=top_feats_3D).map(CHANNEL_PAL)

    g = sns.clustermap(
        plot_3D.T,
        cmap='RdBu_r', center=0, vmin=-3, vmax=3,
        row_colors=row_colors_3D,
        figsize=(8, 14),
        dendrogram_ratio=(0.1, 0.2),
        cbar_pos=(1.02, 0.3, 0.03, 0.3),
        xticklabels=True, yticklabels=True,
    )
    g.ax_heatmap.set_title('3D: Morphological Fingerprints (top 40 features)', pad=12)
    g.ax_heatmap.set_xlabel('Compound')
    save_panel(g.figure, "Fig5f_3D", data=plot_3D,
               caption="3D morphological fingerprints, 5-FU and olaparib",
               notebook="analysis/3_Figure4/3_PairwiseCorrelations.ipynb")
    plt.show()

    # Legend for channel colors
    handles = [Patch(color=v, label=k) for k, v in CHANNEL_PAL.items() if k in channels_3D]
    plt.figure(figsize=(3, 3))
    plt.legend(handles=handles, title='Channel', loc='center', frameon=False)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    print("3D chosen concentrations:")
    for row_label in fp_3D.index:
        print(" ", row_label.replace("\n", " "))
else:
    print("SKIP Fig5f section: normalize_feat unavailable — see KNOWN_ISSUES.md")


### 5-FU: Most Similar Compounds

In [ ]:
def rank_similar_to_fluor(dataset, target_name='Fluor', extra_names=None,
                          grit_threshold=1.96, top_n=15):
    """
    For each drug, pick its lowest grit-passing concentration.
    Compute cosine similarity against 5-FU's profile.
    Return top_n most similar compounds (sorted descending).
    """
    mask = dataset['Metadata_pert_type'] == 'trt'
    if extra_names:
        mask |= (
            (dataset['Metadata_pert_type'] == 'pos_con') &
            dataset['Metadata_name'].isin(extra_names)
        )
    t = dataset[mask].copy()
    feats, _ = list_features(t)

    t_grp = t.groupby('Metadata_pert_name').median(numeric_only=True)
    meta_lookup = (
        t[['Metadata_pert_name', 'Metadata_name', 'Metadata_cmpd_conc']]
        .drop_duplicates('Metadata_pert_name')
        .set_index('Metadata_pert_name')
    )
    name_map = meta_lookup['Metadata_name']
    conc_map = meta_lookup['Metadata_cmpd_conc']

    passing = t_grp[t_grp['Metadata_grit'] > grit_threshold]
    feat_data = passing[feats]
    z = (feat_data - feat_data.mean()) / feat_data.std()
    z['_name'] = passing.index.map(name_map)
    z['_conc'] = passing.index.map(conc_map)

    # pick lowest passing per drug
    best = (
        z.sort_values('_conc')
         .groupby('_name')
         .first()
         .reset_index()
         .set_index('_name')
    )
    if target_name not in best.index:
        print(f"WARNING: {target_name} not found in passing profiles")
        return None

    target_vec  = best.loc[target_name, feats].values.reshape(1, -1)
    other       = best.drop(index=target_name)
    other_vecs  = other[feats].values

    sims = cos_sim(target_vec, other_vecs)[0]
    result = pd.DataFrame({
        'compound':   other.index,
        'conc_µM':    other['_conc'].values,
        'cosine_sim': sims,
    }).sort_values('cosine_sim', ascending=False)

    return result.head(top_n).reset_index(drop=True)


# Rank for 2D and 3D
rank_2D = rank_similar_to_fluor(data_2D,   extra_names=POSCONS)
rank_3D = rank_similar_to_fluor(dataset_pw, extra_names=POSCONS)

print("=== 2D: Most similar to 5-FU ===")
display(rank_2D)

print("\n=== 3D: Most similar to 5-FU ===")
display(rank_3D)


In [ ]:

# Cell B — CHANNEL_PAL, TARGET, POSCONS, get_lowest_passing_profiles
CHANNEL_PAL = {
    'HOECHST':           '#9467bd',
    'PHAandWGA':         '#bcbd22',
    'MITO':              '#8c564b',
    'SYTO':              '#1f77b4',
    'CONC':              '#ff7f0e',
    'AreaShape':         '#2ca02c',
    'Granularity':       '#d62728',
    'Neighbors':         '#e377c2',
    'Correlation':       '#17becf',
    'RadialDistribution':'#aec7e8',
    'Other':             '#7f7f7f',
    'none':              '#7f7f7f',
}

TARGET = {
    'Fluor': '5-FU',
    'Olapa': 'Olaparib',
    'etop':  'Etoposide',
    'Nutli': 'Nutlin',
    'AMG23': 'AMG232',
}
POSCONS = ['etop']

def get_lowest_passing_profiles(dataset, target_keys, extra_names=None, grit_threshold=1.96):
    """
    Returns (fingerprints_df, z_all) where:
    - fingerprints_df: z-scored profile for each drug at its lowest passing concentration
    - z_all:          z-scored profiles for ALL passing concentrations (for ranking)
    """
    mask = dataset['Metadata_pert_type'] == 'trt'
    if extra_names:
        mask |= (
            (dataset['Metadata_pert_type'] == 'pos_con') &
            dataset['Metadata_name'].isin(extra_names)
        )
    t = dataset[mask].copy()
    feats, _ = list_features(t)

    t_grp = t.groupby('Metadata_pert_name').median(numeric_only=True)
    meta_lookup = (
        t[['Metadata_pert_name', 'Metadata_name', 'Metadata_cmpd_conc']]
        .drop_duplicates('Metadata_pert_name')
        .set_index('Metadata_pert_name')
    )
    name_map = meta_lookup['Metadata_name']
    conc_map = meta_lookup['Metadata_cmpd_conc']

    passing = t_grp[t_grp['Metadata_grit'] > grit_threshold]
    feat_data = passing[feats]
    z = (feat_data - feat_data.mean()) / feat_data.std()
    z['_name'] = passing.index.map(name_map)
    z['_conc']  = passing.index.map(conc_map)

    rows = {}
    for key, label in target_keys.items():
        subset = z[z['_name'] == key].sort_values('_conc')
        if len(subset) == 0:
            print(f"WARNING: {label} ({key}) has no concentration passing grit > {grit_threshold}")
            continue
        conc = subset.iloc[0]['_conc']
        rows[f"{label}\n({conc} µM)"] = subset[feats].iloc[0]

    return pd.DataFrame(rows).T, z

print("Cell B: OK")


In [ ]:
# Guarded: this cell is part of the Fig 5f fingerprint comparison, which needs
# `normalize_feat` to match 2D feature names to 3D. That helper is missing from
# the source (see KNOWN_ISSUES.md), so the code is kept verbatim but not run.
if FIG5F_AVAILABLE:

    # Verify 2D fingerprints
    fp_2D, z_2D = get_lowest_passing_profiles(data_2D, TARGET, POSCONS)
    fp_2D.columns = [normalize_feat(c) for c in fp_2D.columns]

    print("fp_2D shape:", fp_2D.shape)
    print("Chosen concentrations (2D):")
    for r in fp_2D.index:
        print(" ", r.replace('\n', ' '))

    print("\nfp_3D check:")
    fp_3D, z_3D = get_lowest_passing_profiles(dataset_pw, TARGET, POSCONS)
    print("fp_3D shape:", fp_3D.shape)
    print("Chosen concentrations (3D):")
    for r in fp_3D.index:
        print(" ", r.replace('\n', ' '))
else:
    print("SKIP Fig5f section: normalize_feat unavailable — see KNOWN_ISSUES.md")


In [ ]:
# Guarded: this cell is part of the Fig 5f fingerprint comparison, which needs
# `normalize_feat` to match 2D feature names to 3D. That helper is missing from
# the source (see KNOWN_ISSUES.md), so the code is kept verbatim but not run.
if FIG5F_AVAILABLE:

    # Cell C — 2D Clustermap
    from matplotlib.patches import Patch

    top_feats_2D  = fp_2D.std(axis=0).nlargest(40).index
    plot_2D       = fp_2D[top_feats_2D]
    channels_2D   = [parse_channel(f) for f in top_feats_2D]
    row_colors_2D = pd.Series(channels_2D, index=top_feats_2D).map(CHANNEL_PAL)

    g = sns.clustermap(
        plot_2D.T,
        cmap='RdBu_r', center=0, vmin=-3, vmax=3,
        row_colors=row_colors_2D,
        figsize=(8, 14),
        dendrogram_ratio=(0.1, 0.2),
        cbar_pos=(1.02, 0.3, 0.03, 0.3),
        xticklabels=True, yticklabels=True,
    )
    g.ax_heatmap.set_title('2D: Morphological Fingerprints (top 40 features)', pad=12)
    g.ax_heatmap.set_xlabel('Compound')
    save_panel(g.figure, "Fig5f_2D", data=plot_2D,
               caption="2D morphological fingerprints, 5-FU and olaparib",
               notebook="analysis/3_Figure4/3_PairwiseCorrelations.ipynb")
    plt.show()

    # Channel legend
    unique_ch = list(dict.fromkeys(channels_2D))
    handles = [Patch(color=CHANNEL_PAL[ch], label=ch) for ch in unique_ch]
    plt.figure(figsize=(3, 2.5))
    plt.legend(handles=handles, title='Channel', loc='center', frameon=False)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    print("2D clustermap: OK — saved to", ImagesOut + 'fingerprints_2D.pdf')
else:
    print("SKIP Fig5f section: normalize_feat unavailable — see KNOWN_ISSUES.md")


In [ ]:
# Guarded: this cell is part of the Fig 5f fingerprint comparison, which needs
# `normalize_feat` to match 2D feature names to 3D. That helper is missing from
# the source (see KNOWN_ISSUES.md), so the code is kept verbatim but not run.
if FIG5F_AVAILABLE:

    # Cell D — 3D Clustermap
    top_feats_3D  = fp_3D.std(axis=0).nlargest(40).index
    plot_3D       = fp_3D[top_feats_3D]
    channels_3D   = [parse_channel(f) for f in top_feats_3D]
    row_colors_3D = pd.Series(channels_3D, index=top_feats_3D).map(CHANNEL_PAL)

    g = sns.clustermap(
        plot_3D.T,
        cmap='RdBu_r', center=0, vmin=-3, vmax=3,
        row_colors=row_colors_3D,
        figsize=(8, 14),
        dendrogram_ratio=(0.1, 0.2),
        cbar_pos=(1.02, 0.3, 0.03, 0.3),
        xticklabels=True, yticklabels=True,
    )
    g.ax_heatmap.set_title('3D: Morphological Fingerprints (top 40 features)', pad=12)
    g.ax_heatmap.set_xlabel('Compound')
    save_panel(g.figure, "Fig5f_3D", data=plot_3D,
               caption="3D morphological fingerprints, 5-FU and olaparib",
               notebook="analysis/3_Figure4/3_PairwiseCorrelations.ipynb")
    plt.show()

    unique_ch_3D = list(dict.fromkeys(channels_3D))
    handles = [Patch(color=CHANNEL_PAL[ch], label=ch) for ch in unique_ch_3D]
    plt.figure(figsize=(3, 2.5))
    plt.legend(handles=handles, title='Channel', loc='center', frameon=False)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    print("3D clustermap: OK — saved to", ImagesOut + 'fingerprints_3D.pdf')
else:
    print("SKIP Fig5f section: normalize_feat unavailable — see KNOWN_ISSUES.md")


In [ ]:

# Cell E — 5-FU ranking
def rank_similar_to_fluor(dataset, target_name='Fluor', extra_names=None,
                          grit_threshold=1.96, top_n=15):
    """
    For each drug, pick its lowest grit-passing concentration.
    Compute cosine similarity against 5-FU's profile.
    Return top_n most similar compounds (sorted descending).
    """
    mask = dataset['Metadata_pert_type'] == 'trt'
    if extra_names:
        mask |= (
            (dataset['Metadata_pert_type'] == 'pos_con') &
            dataset['Metadata_name'].isin(extra_names)
        )
    t = dataset[mask].copy()
    feats, _ = list_features(t)

    t_grp = t.groupby('Metadata_pert_name').median(numeric_only=True)
    meta_lookup = (
        t[['Metadata_pert_name', 'Metadata_name', 'Metadata_cmpd_conc']]
        .drop_duplicates('Metadata_pert_name')
        .set_index('Metadata_pert_name')
    )
    name_map = meta_lookup['Metadata_name']
    conc_map = meta_lookup['Metadata_cmpd_conc']

    passing = t_grp[t_grp['Metadata_grit'] > grit_threshold]
    feat_data = passing[feats]
    z = (feat_data - feat_data.mean()) / feat_data.std()
    z['_name'] = passing.index.map(name_map)
    z['_conc']  = passing.index.map(conc_map)

    # pick lowest passing per drug
    best = (
        z.sort_values('_conc')
         .groupby('_name')
         .first()
         .reset_index()
         .rename(columns={'_name': 'Metadata_name', '_conc': 'conc_µM'})
         .set_index('Metadata_name')
    )

    if target_name not in best.index:
        print(f"WARNING: {target_name} not found in passing profiles")
        return None

    target_vec = best.loc[target_name, feats].values.reshape(1, -1)
    other      = best.drop(index=target_name)
    sims       = cos_sim(target_vec, other[feats].values)[0]

    result = pd.DataFrame({
        'compound':   other.index,
        'conc_µM':    other['conc_µM'].values,
        'cosine_sim': sims,
    }).sort_values('cosine_sim', ascending=False)

    return result.head(top_n).reset_index(drop=True)


rank_2D = rank_similar_to_fluor(data_2D,   extra_names=POSCONS)
rank_3D = rank_similar_to_fluor(dataset_pw, extra_names=POSCONS)

print("=== 2D: Most similar to 5-FU ===")
display(rank_2D)

print("\n=== 3D: Most similar to 5-FU ===")
display(rank_3D)


In [ ]:

# Check kernel state
for v in ['fp_3D', 'z_3D', 'fp_2D', 'z_2D', 'dataset_pw', 'data_2D']:
    try:
        val = eval(v)
        print(f"✓ {v}: {type(val).__name__} {getattr(val, 'shape', '')}")
    except NameError:
        print(f"✗ {v}: not defined")


In [ ]:
# Guarded: this cell is part of the Fig 5f fingerprint comparison, which needs
# `normalize_feat` to match 2D feature names to 3D. That helper is missing from
# the source (see KNOWN_ISSUES.md), so the code is kept verbatim but not run.
# 5-FU only passes grit in the 2D/aggregates runs, not MIP, so these cells
# also need its row to exist before indexing [0].
_HAS_FLUOR = FIG5F_AVAILABLE and any('5-FU' in r for r in fp_3D.index) \
             and any('5-FU' in r for r in fp_2D.index)
if _HAS_FLUOR:

    # 5-FU row label in fp_3D
    fluor_label_3D = [r for r in fp_3D.index if '5-FU' in r][0]
    fluor_label_2D = [r for r in fp_2D.index if '5-FU' in r][0]
    print("3D label:", fluor_label_3D)
    print("2D label:", fluor_label_2D)

    # Top features driving 5-FU's uniqueness in 3D:
    # = features where 5-FU z-score deviates most from all other compounds
    fluor_3D = fp_3D.loc[fluor_label_3D]
    others_3D = fp_3D.drop(index=fluor_label_3D)

    # "uniqueness" = |5-FU z-score| - max |z-score| of any other compound
    # i.e. features where 5-FU sticks out beyond the rest
    deviation = fluor_3D.abs() - others_3D.abs().max(axis=0)
    top_unique_3D = deviation.nlargest(20)

    print("\nTop 20 features most uniquely extreme in 5-FU (3D):")
    for feat, dev in top_unique_3D.items():
        z_val = fluor_3D[feat]
        print(f"  {feat:<55}  5-FU z={z_val:+.2f}  uniqueness={dev:+.2f}")
else:
    print("SKIP Fig5f section: normalize_feat unavailable — see KNOWN_ISSUES.md")


In [ ]:

# Check the range of feature values in dataset_pw to understand if they're already z-scored
feats_pw, _ = list_features(dataset_pw)

# Look at distribution of feature values for a few drugs
sample_feats = feats_pw[:5]
print("Sample feature value ranges in dataset_pw (raw):")
print(dataset_pw[sample_feats].describe().loc[['mean','std','min','max']].round(2))

print("\nFor comparison — after our z-scoring in get_lowest_passing_profiles:")
# reproduce the z-scoring step
mask = dataset_pw['Metadata_pert_type'].isin(['trt', 'pos_con'])
t = dataset_pw[mask].copy()
t_grp = t.groupby('Metadata_pert_name').median(numeric_only=True)
passing = t_grp[t_grp['Metadata_grit'] > 1.96]
feat_data = passing[feats_pw]
z = (feat_data - feat_data.mean()) / feat_data.std()
print(z[sample_feats].describe().loc[['mean','std','min','max']].round(2))


In [ ]:
# Guarded: this cell is part of the Fig 5f fingerprint comparison, which needs
# `normalize_feat` to match 2D feature names to 3D. That helper is missing from
# the source (see KNOWN_ISSUES.md), so the code is kept verbatim but not run.
if FIG5F_AVAILABLE:

    # Build median profiles per drug using only passing concentrations — no extra z-scoring
    def get_median_profiles(dataset, extra_names=None, grit_threshold=1.96):
        mask = dataset['Metadata_pert_type'] == 'trt'
        if extra_names:
            mask |= (dataset['Metadata_pert_type'] == 'pos_con') & dataset['Metadata_name'].isin(extra_names)
        t = dataset[mask].copy()
        feats, _ = list_features(t)
    
        t_grp = t.groupby('Metadata_pert_name').median(numeric_only=True)
        meta_lookup = (
            t[['Metadata_pert_name', 'Metadata_name', 'Metadata_cmpd_conc']]
            .drop_duplicates('Metadata_pert_name').set_index('Metadata_pert_name')
        )
        passing = t_grp[t_grp['Metadata_grit'] > grit_threshold]
        passing = passing.copy()
        passing['_name'] = passing.index.map(meta_lookup['Metadata_name'])
        passing['_conc'] = passing.index.map(meta_lookup['Metadata_cmpd_conc'])
    
        # lowest passing concentration per drug
        best = passing.sort_values('_conc').groupby('_name').first()
        return best[feats], best['_conc']

    prof_3D, conc_3D = get_median_profiles(dataset_pw, POSCONS)
    prof_2D, conc_2D = get_median_profiles(data_2D,    POSCONS)

    # Normalize 2D feature names
    prof_2D.columns = [normalize_feat(c) for c in prof_2D.columns]

    print("3D profiles shape:", prof_3D.shape)
    print("5-FU concentration 3D:", conc_3D.get('Fluor'), "µM")
    print("5-FU concentration 2D:", conc_2D.get('Fluor'), "µM")
else:
    print("SKIP Fig5f section: normalize_feat unavailable — see KNOWN_ISSUES.md")


In [ ]:
# Guarded: this cell is part of the Fig 5f fingerprint comparison, which needs
# `normalize_feat` to match 2D feature names to 3D. That helper is missing from
# the source (see KNOWN_ISSUES.md), so the code is kept verbatim but not run.
if FIG5F_AVAILABLE:

    # Which features are most extreme for 5-FU in 3D vs all other drugs?
    fluor_3D = prof_3D.loc['Fluor']
    others_3D = prof_3D.drop(index='Fluor')

    # Features where 5-FU is most extreme relative to all other compounds
    uniqueness = fluor_3D.abs() - others_3D.abs().max(axis=0)
    top20 = uniqueness.nlargest(20)

    print("Top 20 features most uniquely extreme for 5-FU in 3D")
    print(f"{'Feature':<55} {'5-FU':>7}  {'Others max':>10}  {'Unique':>7}")
    print("-"*85)
    for feat in top20.index:
        z5 = fluor_3D[feat]
        omax = others_3D[feat].abs().max()
        print(f"  {feat:<53} {z5:+7.2f}  {omax:10.2f}  {uniqueness[feat]:+7.2f}")
else:
    print("SKIP Fig5f section: normalize_feat unavailable — see KNOWN_ISSUES.md")


In [ ]:
# Guarded: this cell is part of the Fig 5f fingerprint comparison, which needs
# `normalize_feat` to match 2D feature names to 3D. That helper is missing from
# the source (see KNOWN_ISSUES.md), so the code is kept verbatim but not run.
if FIG5F_AVAILABLE:

    # 1. Where does 5-FU actually have its largest values in 3D?
    top_fluor_3D = fluor_3D.abs().nlargest(15)
    print("5-FU largest absolute feature values in 3D (10 µM):")
    print(f"  {'Feature':<55} {'5-FU 3D':>8}  {'5-FU 2D':>8}")
    print("-"*80)

    # get 2D profile for Fluor
    fluor_2D = prof_2D.loc['Fluor'] if 'Fluor' in prof_2D.index else None

    common = set(prof_3D.columns) & set(prof_2D.columns)
    for feat in top_fluor_3D.index:
        v3 = fluor_3D[feat]
        v2 = prof_2D.loc['Fluor', feat] if feat in common else float('nan')
        print(f"  {feat:<55} {v3:+8.2f}  {v2:+8.2f}")

    print(f"\n5-FU conc: 3D={conc_3D['Fluor']} µM, 2D={conc_2D['Fluor']} µM")
    print(f"Note: 5-FU only passes grit at its highest tested concentration in 3D")
else:
    print("SKIP Fig5f section: normalize_feat unavailable — see KNOWN_ISSUES.md")


In [ ]:
# Guarded: this cell is part of the Fig 5f fingerprint comparison, which needs
# `normalize_feat` to match 2D feature names to 3D. That helper is missing from
# the source (see KNOWN_ISSUES.md), so the code is kept verbatim but not run.
if FIG5F_AVAILABLE:

    # These features don't exist in 2D (NaN) — they're 3D-specific features
    # Let's check: are the top 3D features for 5-FU simply absent from 2D, or is it a real difference?

    top3D_feats = top_fluor_3D.index.tolist()
    in_2D = [f for f in top3D_feats if f in prof_2D.columns]
    not_in_2D = [f for f in top3D_feats if f not in prof_2D.columns]

    print(f"Top 15 features for 5-FU in 3D:")
    print(f"  Present in 2D panel: {len(in_2D)}")
    print(f"  Absent from 2D panel: {len(not_in_2D)}")

    if in_2D:
        print("\nFeatures present in both 2D and 3D:")
        print(f"  {'Feature':<55} {'5-FU 3D':>8}  {'5-FU 2D':>8}")
        for f in in_2D:
            print(f"  {f:<55} {fluor_3D[f]:+8.2f}  {prof_2D.loc['Fluor', f]:+8.2f}")

    # Check overall feature overlap
    print(f"\nOverall feature overlap: {len(common)} / {len(prof_3D.columns)} 3D features present in 2D")
else:
    print("SKIP Fig5f section: normalize_feat unavailable — see KNOWN_ISSUES.md")


In [ ]:
# Guarded: this cell is part of the Fig 5f fingerprint comparison, which needs
# `normalize_feat` to match 2D feature names to 3D. That helper is missing from
# the source (see KNOWN_ISSUES.md), so the code is kept verbatim but not run.
if FIG5F_AVAILABLE:

    # Categorize the top 3D features for 5-FU by measurement type and channel
    print("5-FU top features in 3D — biology summary:")
    print(f"\n{'Feature':<55} {'5-FU':>7}  {'Next highest drug':>20}  {'Next val':>8}")
    print("-"*100)

    top_fluor_3D_20 = fluor_3D.abs().nlargest(20).index
    for feat in top_fluor_3D_20:
        v5 = fluor_3D[feat]
        # which other drug has largest abs value for this feature?
        others_vals = others_3D[feat]
        next_drug = others_vals.abs().idxmax()
        next_val  = others_vals[next_drug]
        print(f"  {feat:<55} {v5:+7.2f}  {next_drug:>20}  {next_val:+8.2f}")

    # Also: does 5-FU have the largest magnitude overall in the 3D dataset?
    print(f"\n5-FU overall L2 norm in 3D: {(fluor_3D**2).sum()**0.5:.2f}")
    for drug in others_3D.index[:5]:
        print(f"{drug} L2 norm: {(others_3D.loc[drug]**2).sum()**0.5:.2f}")
else:
    print("SKIP Fig5f section: normalize_feat unavailable — see KNOWN_ISSUES.md")


## After this added by claude

In [ ]:
import os
print(os.getcwd())


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

dir_data = str(profiles("exp1_main", "")) + "/"

mip_raw = pd.read_parquet(f'{dir_data}grit_data_MIP_HCT116.parquet')
agg_raw = pd.read_parquet(f'{dir_data}grit_data_aggregates_HCT116.parquet')

print("MIP shape:", mip_raw.shape)
print("AGG shape:", agg_raw.shape)

In [ ]:
# NOTE (port): this cell defines get_sim_matrix/shared and originally sat AFTER
# the cell that uses them — it only worked in a live kernel. Moved ahead of it.
def list_features(data):
    meta_cols = [c for c in data.columns if c.startswith('Metadata_')]
    feat_cols = [c for c in data.columns if not c.startswith('Metadata_')]
    return meta_cols, feat_cols

def get_sim_matrix(dataset, grit_threshold=1.96, filter_to=None):
    df = dataset[dataset['Metadata_grit'] > grit_threshold].copy()
    if filter_to is not None:
        df = df[df['Metadata_name'].isin(filter_to)]
    _, feat_cols = list_features(df)
    data = df.groupby('Metadata_name')[feat_cols].mean()
    sim = pd.DataFrame(cosine_similarity(data), index=data.index, columns=data.index)
    return sim

# Drugs passing grit in each
mip_drugs = set(mip_raw[mip_raw['Metadata_grit'] > 1.96]['Metadata_name'].unique())
agg_drugs = set(agg_raw[agg_raw['Metadata_grit'] > 1.96]['Metadata_name'].unique())
shared = mip_drugs & agg_drugs

print(f"MIP passing grit: {len(mip_drugs)}")
print(f"AGG passing grit: {len(agg_drugs)}")
print(f"Shared (intersection): {len(shared)}")
print(sorted(shared))

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import silhouette_score, adjusted_rand_score

# Drug → pathway mapping
drug_to_pathway = (
    mip_raw[mip_raw['Metadata_name'].isin(shared)]
    .drop_duplicates('Metadata_name')
    .set_index('Metadata_name')['Metadata_pathway']
    .to_dict()
)

# At least one shared drug has no pathway annotation, and adjusted_rand_score
# cannot sort a label list mixing None with strings. Two cells below the notebook
# reaches the same conclusion and rebuilds these matrices from `shared_annotated`;
# applying it here too keeps this cell consistent with that, instead of raising.
shared = {d for d in shared if drug_to_pathway.get(d) is not None}

# Build similarity matrices for shared drugs
sim_mip = get_sim_matrix(mip_raw, filter_to=shared)
sim_agg = get_sim_matrix(agg_raw, filter_to=shared)

def cluster_metrics(sim_matrix, drug_to_pathway, k_range=range(2, 9)):
    dist = (1 - sim_matrix).clip(lower=0).values
    Z = linkage(dist, method='ward')
    pathway_labels = [drug_to_pathway[d] for d in sim_matrix.index]
    rows = []
    for k in k_range:
        labels = fcluster(Z, t=k, criterion='maxclust')
        sil = silhouette_score(dist, labels, metric='precomputed')
        ari = adjusted_rand_score(pathway_labels, labels)
        rows.append({'k': k, 'silhouette': round(sil, 3), 'ARI': round(ari, 3)})
    return pd.DataFrame(rows)

metrics_mip = cluster_metrics(sim_mip, drug_to_pathway)
metrics_agg = cluster_metrics(sim_agg, drug_to_pathway)

print("=== MIP ===")
print(metrics_mip.to_string(index=False))
print("\n=== scAgg ===")
print(metrics_agg.to_string(index=False))

In [ ]:
# Check for missing pathway labels
print("Drugs with missing pathway:")
for d in sorted(shared):
    pw = drug_to_pathway.get(d)
    if pw is None:
        print(f"  {d}: {pw}")

In [ ]:
# Exclude drugs with no pathway annotation
shared_annotated = {d for d in shared if drug_to_pathway.get(d) is not None}
print(f"Shared annotated drugs: {len(shared_annotated)}")

sim_mip = get_sim_matrix(mip_raw, filter_to=shared_annotated)
sim_agg = get_sim_matrix(agg_raw, filter_to=shared_annotated)

metrics_mip = cluster_metrics(sim_mip, drug_to_pathway)
metrics_agg = cluster_metrics(sim_agg, drug_to_pathway)

print("\n=== MIP ===")
print(metrics_mip.to_string(index=False))
print("\n=== scAgg ===")
print(metrics_agg.to_string(index=False))

In [ ]:
from scipy.spatial.distance import squareform
import warnings

def cluster_metrics(sim_matrix, drug_to_pathway, k_range=range(2, 9)):
    # pandas >=2 copy-on-write makes .values read-only, so the in-place
    # fill_diagonal below raises. Same fix the port applied in
    # calculate_similar_pairs; see KNOWN_ISSUES.md.
    dist_sq = (1 - sim_matrix).clip(lower=0).to_numpy(dtype=float, copy=True)
    np.fill_diagonal(dist_sq, 0)
    dist_condensed = squareform(dist_sq)
    Z = linkage(dist_condensed, method='ward')
    pathway_labels = [drug_to_pathway[d] for d in sim_matrix.index]
    rows = []
    for k in k_range:
        labels = fcluster(Z, t=k, criterion='maxclust')
        sil = silhouette_score(dist_sq, labels, metric='precomputed')
        ari = adjusted_rand_score(pathway_labels, labels)
        rows.append({'k': k, 'silhouette': round(sil, 3), 'ARI': round(ari, 3)})
    return pd.DataFrame(rows)

metrics_mip = cluster_metrics(sim_mip, drug_to_pathway)
metrics_agg = cluster_metrics(sim_agg, drug_to_pathway)

print("=== MIP (n=38 drugs, grit>1.96, annotated) ===")
print(metrics_mip.to_string(index=False))
k_opt_mip = metrics_mip.loc[metrics_mip['silhouette'].idxmax()]
print(f"\n→ Optimal k = {int(k_opt_mip.k)}, silhouette = {k_opt_mip.silhouette}, ARI = {k_opt_mip.ARI}")

print("\n=== scAgg (n=38 drugs, grit>1.96, annotated) ===")
print(metrics_agg.to_string(index=False))
k_opt_agg = metrics_agg.loc[metrics_agg['silhouette'].idxmax()]
print(f"\n→ Optimal k = {int(k_opt_agg.k)}, silhouette = {k_opt_agg.silhouette}, ARI = {k_opt_agg.ARI}")

In [ ]:
import os

out_dir = str(ROOT / 'analysis' / '3_Figure4' / 'results')
os.makedirs(out_dir, exist_ok=True)

metrics_mip['data_type'] = 'MIP'
metrics_agg['data_type'] = 'scAgg'

combined = pd.concat([metrics_mip, metrics_agg])
combined.to_csv(f'{out_dir}/hierarchical_cluster_metrics_HCT116.csv', index=False)
print("Saved to", f'{out_dir}/hierarchical_cluster_metrics_HCT116.csv')
print()
print(combined.to_string(index=False))

In [ ]:
import os
# Check what result files exist
for f in os.listdir(str(figdir('Fig4'))):
    print(f)